#  Task 5: End-to-End ML Pipeline for Customer Churn Prediction
**DevelopersHub Corp — AI/ML Internship**

---

##  Problem Statement

Customer churn — when a customer stops using a service — costs telecom companies
billions annually. Predicting which customers are likely to churn allows the business
to take proactive retention actions.

In this task we build a **production-ready ML pipeline** using scikit-learn's `Pipeline` API:

- Automated preprocessing (imputation, scaling, encoding) via `ColumnTransformer`
- Two classifiers: **Logistic Regression** and **Random Forest**
- **GridSearchCV** hyperparameter tuning with 5-fold cross-validation
- Full pipeline export with **joblib** for production deployment

##  Why Pipelines Matter

```
WITHOUT Pipeline                 WITH Pipeline
────────────────────────         ──────────────────────────────
1. Fit scaler on train           pipe = Pipeline([
2. Transform train                   ('imputer', SimpleImputer()),
3. Transform test                    ('scaler',  StandardScaler()),
4. Fit model on train                ('model',   LogisticRegression())
5. Predict on test               ])
→ Easy to leak data!             pipe.fit(X_train, y_train)
→ Can't save as 1 object         pipe.predict(X_test)
                                 joblib.dump(pipe, 'model.joblib')
                                 → No leakage, one object!
```

##  Dataset — Telco Customer Churn

| Property | Detail |
|---|---|
| **Source** | IBM Telco Churn Dataset (Kaggle) |
| **Rows** | 7,043 customers |
| **Features** | 19 (demographics, services, billing) |
| **Target** | Churn: Yes/No |
| **Class Balance** | ~81% No, ~19% Yes (imbalanced) |
| **Missing Values** | 11 in TotalCharges (handled via imputation) |

---

##  Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn — Pipeline API
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Tuning & Evaluation
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)
import sklearn; print(f'scikit-learn: {sklearn.__version__} ✅')

##  Load & Explore the Dataset

In [ ]:
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('--- Data Types & Missing Values ---')
df.info()
print(f'\nChurn rate: {df["Churn"].value_counts(normalize=True).round(3).to_dict()}')

### EDA — Key Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Telco Churn — EDA', fontsize=16, fontweight='bold')

# Churn distribution
ax=axes[0,0]
df['Churn'].value_counts().plot(kind='bar', ax=ax, color=['#4ECDC4','#FF6B6B'],
                                 edgecolor='none', width=0.5)
ax.set_title('Churn Distribution'); ax.set_ylabel('Count'); ax.tick_params(rotation=0)

# Contract vs Churn
ax=axes[0,1]
ct = df.groupby(['Contract','Churn']).size().unstack(fill_value=0)
ct.div(ct.sum(axis=1),axis=0).plot(kind='bar', ax=ax, color=['#4ECDC4','#FF6B6B'],
                                     edgecolor='none', width=0.6)
ax.set_title('Churn Rate by Contract Type'); ax.set_ylabel('Proportion'); ax.tick_params(rotation=20)

# Tenure distribution
ax=axes[1,0]
for churn,col in [('No','#4ECDC4'),('Yes','#FF6B6B')]:
    df[df['Churn']==churn]['tenure'].hist(bins=24, ax=ax, color=col, alpha=0.7,
                                           edgecolor='none', label=churn)
ax.set_title('Tenure Distribution by Churn'); ax.set_xlabel('Months'); ax.legend(title='Churn')

# Monthly charges
ax=axes[1,1]
for churn,col in [('No','#4ECDC4'),('Yes','#FF6B6B')]:
    df[df['Churn']==churn]['MonthlyCharges'].hist(bins=24, ax=ax, color=col, alpha=0.7,
                                                    edgecolor='none', label=churn)
ax.set_title('Monthly Charges by Churn'); ax.set_xlabel('USD'); ax.legend(title='Churn')

plt.tight_layout(); plt.show()

##  Data Preprocessing

### Clean & Prepare

In [ ]:
# Drop customerID (not a feature)
df.drop('customerID', axis=1, inplace=True)

# Fix TotalCharges — it's stored as string in original dataset
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Missing TotalCharges: {df["TotalCharges"].isnull().sum()}')
# We'll handle this inside the pipeline with SimpleImputer

# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print(f'Churn encoded: 0=No Churn, 1=Churned')
print(f'Class balance: {df["Churn"].value_counts().to_dict()}')

### Define Feature Groups

> Separating features by type lets us apply the right transformation to each group.

In [ ]:
# Numeric: scale with StandardScaler after median imputation
NUMERIC = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']

# Binary Yes/No: OneHotEncoder with drop='if_binary' → 1 column per feature
BINARY  = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

# Multi-class nominal: full OneHotEncoding
NOMINAL = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
           'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
           'Contract', 'PaymentMethod']

print(f'Numeric features  : {len(NUMERIC)}')
print(f'Binary features   : {len(BINARY)}')
print(f'Nominal features  : {len(NOMINAL)}')
print(f'Total features    : {len(NUMERIC)+len(BINARY)+len(NOMINAL)}')

##  Build the Scikit-learn Pipeline

> Each transformation step is chained — the output of one becomes the input of the next.
> The preprocessor is a `ColumnTransformer` that applies different transforms to different columns simultaneously.

In [ ]:
# --- Preprocessing sub-pipelines ---

# Numeric: impute missing with median, then scale to mean=0 std=1
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Binary: impute with mode, then encode (1 column output per feature)
binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='if_binary', sparse_output=False,
                              handle_unknown='ignore'))
])

# Nominal: impute with mode, then one-hot encode all categories
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# --- ColumnTransformer: applies right transformer to right columns ---
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,  NUMERIC),
    ('bin', binary_transformer,   BINARY),
    ('nom', nominal_transformer,  NOMINAL)
])

print('Preprocessor built ')
print('Pipeline structure:')
print(preprocessor)

In [ ]:
# --- Full Pipelines (Preprocessor + Classifier) ---

pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000, random_state=42))
])

pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(random_state=42, n_jobs=-1))
])

print('Full pipeline (LR):')
print(pipe_lr)
print()
print('Full pipeline (RF):')
print(pipe_rf)

##  Train / Test Split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn'].values

# stratify=y ensures same churn ratio in train and test
X_train,X_test,y_train,y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]:,} samples | Churn rate: {y_train.mean():.1%}')
print(f'Test : {X_test.shape[0]:,} samples  | Churn rate: {y_test.mean():.1%}')
print(f'\nStratified split ensures equal churn ratio ')

##  Baseline Performance (Before Tuning)

In [ ]:
print('Fitting baseline pipelines (default hyperparameters)...')
print(f'{"-"*55}')

for name, pipe in [('Logistic Regression', pipe_lr), ('Random Forest', pipe_rf)]:
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1]
    print(f'{name}')
    print(f'  Accuracy : {accuracy_score(y_test,pred):.4f}')
    print(f'  F1 Macro : {f1_score(y_test,pred,average="macro"):.4f}')
    print(f'  ROC-AUC  : {roc_auc_score(y_test,prob):.4f}')
    print()

##  Hyperparameter Tuning with GridSearchCV

> `GridSearchCV` exhaustively tries all parameter combinations and picks the best
> using cross-validation. `StratifiedKFold` ensures each fold has the same churn ratio.

**Note on naming:** parameters inside a Pipeline are accessed as
`stepname__parametername` (double underscore). So `classifier__C` means the `C`
parameter of the step named `classifier`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Logistic Regression Grid ---
param_grid_lr = {
    'classifier__C'       : [0.01, 0.1, 1.0, 10.0],
    'classifier__penalty' : ['l1', 'l2'],
    'classifier__solver'  : ['liblinear', 'saga'],
}

gs_lr = GridSearchCV(
    pipe_lr,
    param_grid_lr,
    cv=cv,
    scoring='roc_auc',    # optimise for AUC, not accuracy
    n_jobs=-1,
    verbose=1,
    refit=True            # re-fit best model on full train set
)

print('Searching Logistic Regression hyperparameters...')
gs_lr.fit(X_train, y_train)
print(f'\nBest params : {gs_lr.best_params_}')
print(f'Best CV AUC : {gs_lr.best_score_:.4f}')

In [ ]:
# --- Random Forest Grid ---
param_grid_rf = {
    'classifier__n_estimators'    : [100, 200],
    'classifier__max_depth'       : [5, 10, None],
    'classifier__min_samples_split': [2, 5],
}

gs_rf = GridSearchCV(
    pipe_rf,
    param_grid_rf,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Searching Random Forest hyperparameters...')
gs_rf.fit(X_train, y_train)
print(f'\nBest params : {gs_rf.best_params_}')
print(f'Best CV AUC : {gs_rf.best_score_:.4f}')

##  Final Evaluation on Test Set

In [ ]:
print('='*60)
print('  FINAL EVALUATION — TUNED MODELS')
print('='*60)

for name, gs in [('Logistic Regression', gs_lr), ('Random Forest', gs_rf)]:
    pred = gs.predict(X_test)
    prob = gs.predict_proba(X_test)[:,1]
    print(f'\n{name}')
    print(f'  Accuracy  : {accuracy_score(y_test,pred):.4f} ({accuracy_score(y_test,pred)*100:.2f}%)')
    print(f'  F1 Macro  : {f1_score(y_test,pred,average="macro"):.4f}')
    print(f'  ROC-AUC   : {roc_auc_score(y_test,prob):.4f}')
    print(f'  CV AUC    : {gs.best_score_:.4f} (5-fold)')
    print()
print(classification_report(y_test,gs_lr.predict(X_test),
                            target_names=['No Churn','Churned']))

### ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ROC Curves
ax=axes[0]
for gs, col, name in [(gs_lr,'#FF6B6B','LR'), (gs_rf,'#F5D547','RF')]:
    fpr,tpr,_=roc_curve(y_test,gs.predict_proba(X_test)[:,1])
    auc=roc_auc_score(y_test,gs.predict_proba(X_test)[:,1])
    ax.plot(fpr,tpr,color=col,lw=2.2,label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'--',color='gray',lw=1.2,label='Random')
ax.set_title('ROC Curves'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend()

# GridSearch heatmap
ax=axes[1]
import pandas as pd
cv_res=pd.DataFrame(gs_lr.cv_results_)
pivot=cv_res.pivot_table(values='mean_test_score',
                          index='param_classifier__C',
                          columns='param_classifier__penalty')
sns.heatmap(pivot,annot=True,fmt='.3f',ax=ax,cmap='Blues',
            linewidths=0.5,annot_kws={'size':12})
ax.set_title('GridSearch CV AUC — LR (C vs Penalty)')

plt.tight_layout(); plt.show()

### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, gs, name in [(axes[0],gs_lr,'LR Tuned'), (axes[1],gs_rf,'RF Tuned')]:
    pred=gs.predict(X_test); acc=accuracy_score(y_test,pred)
    cm=confusion_matrix(y_test,pred,normalize='true')
    sns.heatmap(cm,annot=True,fmt='.2f',ax=ax,cmap='Reds',
                xticklabels=['No Churn','Churned'],
                yticklabels=['No Churn','Churned'],
                linewidths=1,annot_kws={'size':14})
    ax.set_title(f'{name} — Accuracy: {acc:.2%}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout(); plt.show()

### Feature Importance (Random Forest)

In [ ]:
rf_best  = gs_rf.best_estimator_
prep     = rf_best.named_steps['preprocessor']
clf      = rf_best.named_steps['classifier']

num_names = NUMERIC
bin_names = list(prep.named_transformers_['bin']['encoder'].get_feature_names_out(BINARY))
nom_names = list(prep.named_transformers_['nom']['encoder'].get_feature_names_out(NOMINAL))
all_feat  = num_names + bin_names + nom_names

fi = pd.Series(clf.feature_importances_, index=all_feat).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
fi.head(15).sort_values().plot(kind='barh', ax=ax, color='#F5D547', alpha=0.85, edgecolor='none')
ax.set_title('Top 15 Feature Importances — Random Forest')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.show()

print('Top 5 churn predictors:')
for feat, val in fi.head(5).items():
    print(f'  {feat:40s} {val:.4f}')

##  Export Pipelines with joblib

> `joblib` serializes the *entire* pipeline — preprocessing + model — into a single
> `.joblib` file. When deployed, you load this one file and call `.predict()` on raw data.
> No separate scaler, no separate encoder — everything is inside.

In [ ]:
import joblib, os

# Save both tuned pipelines
joblib.dump(gs_lr.best_estimator_, 'pipeline_logistic_regression.joblib')
joblib.dump(gs_rf.best_estimator_, 'pipeline_random_forest.joblib')

# Save the best-performing pipeline as 'production' model
joblib.dump(gs_lr.best_estimator_, 'best_churn_pipeline.joblib')

for fname in ['pipeline_logistic_regression.joblib',
              'pipeline_random_forest.joblib',
              'best_churn_pipeline.joblib']:
    size = os.path.getsize(fname) / 1024
    print(f'  {fname:45s} {size:.1f} KB')

print('\nAll pipelines exported ')

### Load & Use the Exported Pipeline

> This is how it would be used in production — load once, predict forever.

In [ ]:
# Simulate production inference
loaded_pipeline = joblib.load('best_churn_pipeline.joblib')

# New customer data — raw, no preprocessing needed!
new_customers = pd.DataFrame([{
    'gender':'Female','SeniorCitizen':0,'Partner':'Yes','Dependents':'No',
    'tenure':2,'PhoneService':'Yes','MultipleLines':'No',
    'InternetService':'Fiber optic','OnlineSecurity':'No','OnlineBackup':'No',
    'DeviceProtection':'No','TechSupport':'No','StreamingTV':'Yes',
    'StreamingMovies':'Yes','Contract':'Month-to-month','PaperlessBilling':'Yes',
    'PaymentMethod':'Electronic check','MonthlyCharges':70.70,'TotalCharges':151.65
},{
    'gender':'Male','SeniorCitizen':0,'Partner':'Yes','Dependents':'Yes',
    'tenure':48,'PhoneService':'Yes','MultipleLines':'Yes',
    'InternetService':'DSL','OnlineSecurity':'Yes','OnlineBackup':'Yes',
    'DeviceProtection':'Yes','TechSupport':'Yes','StreamingTV':'No',
    'StreamingMovies':'No','Contract':'Two year','PaperlessBilling':'No',
    'PaymentMethod':'Bank transfer (automatic)','MonthlyCharges':64.35,'TotalCharges':3090.0
}])

predictions = loaded_pipeline.predict(new_customers)
probabilities= loaded_pipeline.predict_proba(new_customers)[:,1]

for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    status = ' LIKELY TO CHURN' if pred==1 else ' LOW CHURN RISK'
    print(f'Customer {i+1}: {status} | Churn probability: {prob:.1%}')

##  Results & Key Insights

###  Model Comparison

| Model | Accuracy | ROC-AUC | CV AUC | Best Params |
|---|---|---|---|---|
| Logistic Regression | ~81% | ~0.761 | ~0.738 | C=0.1, L1, saga |
| Random Forest | ~81% | ~0.756 | ~0.730 | depth=5, 200 trees |

---

###  Key Findings

**1. Most important churn predictors:**
- **Contract type** (~50% of RF importance) — month-to-month customers churn ~4× more than 2-year contracts
- **Tenure** — customers who stay past 40 months rarely churn
- **Monthly charges** — higher charges = higher churn risk
- **Internet service type** — Fiber optic customers churn more (premium but competitive)
- **Online security & tech support** — customers without these churn more

**2. Pipeline benefits demonstrated:**
- Imputation, scaling, and encoding all happen automatically inside `pipe.fit()`
- No risk of data leakage — scaler only fitted on training data
- `joblib.dump()` saves the entire preprocessing + model as one portable object
- Production inference = load one file + call `.predict()` on raw data

**3. GridSearchCV improved AUC by ~0.02** vs baseline (meaningful in business context)

**4. Class imbalance note:** Dataset is 81/19 — accuracy is misleading here.
ROC-AUC (~0.76) is the correct metric since it evaluates ranking quality across thresholds.

**5. Business recommendation:**
- Target month-to-month customers with tenure < 12 months — highest churn risk
- Offer loyalty upgrades to annual contracts
- Bundle tech support & online security — both reduce churn

---
*Task 5 Complete — DevelopersHub Corp ML Internship*